In [115]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [116]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [117]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [118]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [119]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [120]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [121]:
def tokenize_lematize(tekst):
    tokens = word_tokenize(tekst.lower())  # vse besede pišemo z malo začetnico
    tagged = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word.isalpha()    # znebimo se ločil, st,...
    ]

In [122]:
from sklearn.feature_extraction import text

In [123]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()

custom_words = {
    'book', 'novel', 'story', 'reader',
    'edition', 'classic', 'introduction',
    'publish', 'note', 'cover', 'series',
    'time', 'year', 'new', 'make', 'tell',
    'begin', 'just', 'work', 'face',
    'place', 'mean', 'text', 'author', 'original', 'u', 'seller',
    'masterpiece', 'literature', 'best', 'read', 'man', 'men', 'woman', 'life',
    'fiction', 'tale'
}

all_stopwords = list(base_stopwords.union(custom_words))

In [124]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)


# vzamemo 49/50 knjig, eno bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\dipl_data\knjige_opisi\*.txt')
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
vectorizer= CountVectorizer(stop_words= all_stopwords, #custom_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=2, 
                            max_df=10)

In [125]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['far'] not in stop_words.
  warnings.warn(


In [126]:
words = vectorizer.get_feature_names_out()
freq = np.asarray(X.sum(axis=0)).flatten()

top_words = [words[i] for i in freq.argsort()[-50:]]
print(top_words)

['terrible', 'dangerous', 'country', 'generation', 'return', 'rule', 'evil', 'freedom', 'far', 'consequence', 'come', 'sense', 'seek', 'father', 'game', 'lose', 'learn', 'leave', 'live', 'set', 'murder', 'remain', 'jane', 'alternate', 'wizard', 'grow', 'friendship', 'include', 'friend', 'home', 'turn', 'student', 'boy', 'force', 'katniss', 'ring', 'school', 'potter', 'great', 'age', 'power', 'voldemort', 'war', 'family', 'lord', 'young', 'death', 'dark', 'hogwarts', 'harry']


In [127]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1338 stored elements and shape (49, 466)>
  Coords	Values
  (0, 328)	1
  (0, 444)	2
  (0, 154)	2
  (0, 40)	2
  (0, 447)	1
  (0, 407)	1
  (0, 297)	1
  (0, 459)	1
  (0, 120)	1
  (0, 163)	1
  (0, 113)	1
  (0, 274)	1
  (0, 319)	1
  (0, 7)	1
  (0, 349)	1
  (0, 384)	1
  (0, 307)	1
  (0, 209)	1
  (0, 207)	1
  (0, 178)	1
  (0, 335)	1
  (0, 25)	1
  (0, 85)	1
  (0, 104)	1
  (0, 422)	1
  :	:
  (48, 205)	1
  (48, 359)	1
  (48, 374)	1
  (48, 300)	2
  (48, 342)	1
  (48, 246)	1
  (48, 286)	1
  (48, 99)	1
  (48, 208)	1
  (48, 98)	1
  (48, 188)	1
  (48, 367)	1
  (48, 293)	1
  (48, 29)	2
  (48, 196)	1
  (48, 114)	1
  (48, 76)	1
  (48, 437)	1
  (48, 169)	2
  (48, 337)	1
  (48, 109)	1
  (48, 232)	1
  (48, 122)	1
  (48, 27)	1
  (48, 368)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 1 0]
 ...
 [1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   acclaim  achieve  achievement  act  action  adult  adventure  affair  \
0        0      

In [128]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Število komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction error za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break
    return W, H, errors

In [129]:
# test na 'slabem' datasetu

W, H, errors = nmf(X, 4)
print(errors)
#print(W)
#print(H)

[np.float64(41.77200297429941), np.float64(40.34639123213188), np.float64(39.62743353908084), np.float64(38.76306352958167), np.float64(38.03587287827769), np.float64(37.56901473701837), np.float64(37.26113204050538), np.float64(37.041779455598075), np.float64(36.886453643168636), np.float64(36.78579422981352), np.float64(36.72087756779863), np.float64(36.67490387145833), np.float64(36.64303212515492), np.float64(36.61927833704045), np.float64(36.600874520121394), np.float64(36.58769597518421), np.float64(36.57783405442564), np.float64(36.56869383741617), np.float64(36.55852462940374), np.float64(36.549735436879594), np.float64(36.543305500293435), np.float64(36.538816836734156), np.float64(36.53554473226445), np.float64(36.53298841979922), np.float64(36.53071017650973), np.float64(36.52905543411964), np.float64(36.52783125624048), np.float64(36.52685199389325), np.float64(36.526104028639026), np.float64(36.52548868418403), np.float64(36.52497652811495), np.float64(36.5245575295545), n

In [130]:
""" # iskanje najbolj ugodne kombinacije min_df in max_df

min_dfs = [2, 3, 4, 5]

results = []

for min_df in min_dfs:
        vectorizer= CountVectorizer(stop_words= 'english',  
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=min_df)
        X = vectorizer.fit_transform(filepaths)

        W, H, errors = nmf(X, 4)
        results.append({
            "min_df": min_df,
            "error": errors[-1],
            "n_features": X.shape[1]
            
        })
        feature_names = vectorizer.get_feature_names_out()
        for topic_idx, topic in enumerate(H):
            top_words = [feature_names[i] for i in topic.argsort()[-15:]]
            print(f"Tema {topic_idx+1}: {' '.join(top_words)}")
            
for r in results:
    print(r) """
       

' # iskanje najbolj ugodne kombinacije min_df in max_df\n\nmin_dfs = [2, 3, 4, 5]\n\nresults = []\n\nfor min_df in min_dfs:\n        vectorizer= CountVectorizer(stop_words= \'english\',  \n                            tokenizer= tokenize_lematize,\n                            input = \'filename\', \n                            encoding=\'latin-1\', \n                            min_df=min_df)\n        X = vectorizer.fit_transform(filepaths)\n\n        W, H, errors = nmf(X, 4)\n        results.append({\n            "min_df": min_df,\n            "error": errors[-1],\n            "n_features": X.shape[1]\n\n        })\n        feature_names = vectorizer.get_feature_names_out()\n        for topic_idx, topic in enumerate(H):\n            top_words = [feature_names[i] for i in topic.argsort()[-15:]]\n            print(f"Tema {topic_idx+1}: {\' \'.join(top_words)}")\n\nfor r in results:\n    print(r) '

In [131]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-15:]]
    print(f"Tema {topic_idx+1}: {' '.join(top_words)}")

Tema 1: forge darkness crack journey remain age young hobbit rule power dark lord baggins bilbo ring
Tema 2: suzanne lose collins cruel second death turn rebellion win everdeen district hunger capitol game katniss
Tema 3: family charlotte friendship american freedom leave brontã alternate home young secret power force war jane
Tema 4: witchcraft evil turn friend wizarding student school death dark wizard lord potter voldemort hogwarts harry
